# picoNewton_v4 — Full-Resolution Colab Workflow

This notebook is the single executable entry point for the six-artery anisotropic Womersley → membrane → Piezo1 calculation. Run all cells once from top to bottom.

It mounts Google Drive in Colab, creates a unique timestamped run directory, checks out the pinned package commit, verifies the package import in the active kernel, executes the full-resolution six-artery workflow, generates decisions and figures, and archives the complete computational record.

**No reduced-resolution simulation or package test suite is run.** The scientific outputs remain conditional on the packaged model assumptions and calibration status.


## Scientific question
Does anisotropic near-wall forcing provide mechanosensory information distinct from wall shear stress across Aortic Root, Thoracic Aorta, Femoral, Carotid, Iliac, and Brachial arteries?

The workflow keeps wall shear stress, signed transverse Lamb force, and nonnegative Lamb-force exposure physically separate. Signed force is directional; exposure is never treated as a signed traction.


## Vector-resolved membrane and Piezo1 model
Normal and tangential mechanics are transferred through separate passive viscoelastic branches to apical and junctional membrane tension. The four-state Piezo1 model reports open probability, current, and a reduced-order calcium-scale endpoint. Predictions remain conditional on the packaged inputs, constitutive assumptions, transfer parameters, and calibration.


## Fixed production resolution
The calculation is fixed at radial order 150, 2,048 time points per cardiac cycle, 256 near-wall integration points, six arteries, the complete control matrix, and the configured sensitivity analysis. The package API names this resolution `full`; it is not a user-selectable mode here.


In [ ]:
try:
    from google.colab import drive
except ModuleNotFoundError:
    IN_COLAB = False
    print('Non-Colab execution detected; using a local runtime directory.')
else:
    IN_COLAB = True
    drive.mount('/content/drive')


In [ ]:
REPO_URL='https://github.com/khalid-saqr/picoNewton.git'
REPO_REF='8fd376075e90a739ca74f1afede8a895344b3054'
PACKAGE_SUBDIR='picoNewton_v4'
CALIBRATION_RELATIVE_PATH='configs/literature_calibration.json'
NUMERICAL_PROFILE='full'
RUN_SENSITIVITY_SCAN=True
MINIMUM_PASSING_ARTERIES=4
EXPECTED_ARTERIES={'Aortic Root','Thoracic Aorta','Femoral','Carotid','Iliac','Brachial'}


## Timestamped runtime

In Colab, each execution writes to:

`MyDrive/picoNewton_v4_runtime/runs/YYYYMMDD_HHMMSS_UTC_<suffix>/`

Heavy computation uses local runtime storage and the completed output is copied to Drive. Existing runs are never overwritten. Outside Colab, the same notebook uses `PICONEWTON_RUNTIME_ROOT` or a temporary local directory; this supports automated end-to-end validation without changing the Colab behavior.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import secrets
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '2'

RUN_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')}_{secrets.token_hex(2)}"

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive/picoNewton_v4_runtime')
    LOCAL_BASE = Path('/content/picoNewton_v4_work')
else:
    DRIVE_ROOT = Path(os.environ.get('PICONEWTON_RUNTIME_ROOT', '/tmp/picoNewton_v4_runtime'))
    LOCAL_BASE = Path(os.environ.get('PICONEWTON_LOCAL_ROOT', '/tmp/picoNewton_v4_work'))

RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
LOCAL_ROOT = LOCAL_BASE / RUN_ID
REPO_DIR = LOCAL_ROOT / 'repository'
PACKAGE_DIR = REPO_DIR / PACKAGE_SUBDIR
LOCAL_OUTPUT = LOCAL_ROOT / 'model_outputs'

LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
for path in (
    RUN_DIR / 'inputs',
    RUN_DIR / 'configs',
    RUN_DIR / 'logs',
    RUN_DIR / 'provenance',
    RUN_DIR / 'environment',
):
    path.mkdir(parents=True, exist_ok=True)

print('Run ID:', RUN_ID)
print('Persistent run directory:', RUN_DIR)
print('Local work directory:', LOCAL_ROOT)


## Install and verify the pinned package

The exact Git commit is checked out and installed as a normal package. Because the repository uses a `src/` layout, the source directory is also inserted explicitly into the active kernel before import. The same cell imports the package and its workflow modules, verifies version `4.0.0`, and records the module path. Any failure stops execution before scientific computation begins.


In [ ]:
def run_logged(command, log_path, *, cwd=None):
    command = [str(item) for item in command]
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    _ = Path(log_path).write_text(
        '$ ' + ' '.join(command) + '\n\n' + completed.stdout,
        encoding='utf-8',
    )
    print(completed.stdout[-4000:])
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with code {completed.returncode}: {command}')
    return completed.stdout

run_logged(
    ['git', 'clone', '--no-checkout', REPO_URL, REPO_DIR],
    RUN_DIR / 'logs' / 'git_clone.log',
)
run_logged(
    ['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', REPO_REF],
    RUN_DIR / 'logs' / 'git_fetch.log',
)
run_logged(
    ['git', '-C', REPO_DIR, 'checkout', '--detach', 'FETCH_HEAD'],
    RUN_DIR / 'logs' / 'git_checkout.log',
)

GIT_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    text=True,
).strip()
if GIT_SHA != REPO_REF:
    raise RuntimeError(f'Commit mismatch: expected {REPO_REF}, resolved {GIT_SHA}')
if not PACKAGE_DIR.is_dir():
    raise FileNotFoundError(f'Package directory not found: {PACKAGE_DIR}')

run_logged(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--force-reinstall',
        '--no-deps',
        '--no-build-isolation',
        str(PACKAGE_DIR),
    ],
    RUN_DIR / 'logs' / 'pip_install.log',
)

SOURCE_DIR = PACKAGE_DIR / 'src'
PACKAGE_INIT = SOURCE_DIR / 'piconewton_v4' / '__init__.py'
if not PACKAGE_INIT.is_file():
    raise FileNotFoundError(f'src-layout package missing: {PACKAGE_INIT}')

source_path = str(SOURCE_DIR.resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)

# Prevent a stale module from a previous notebook attempt from being reused.
for module_name in list(sys.modules):
    if module_name == 'piconewton_v4' or module_name.startswith('piconewton_v4.'):
        del sys.modules[module_name]

import importlib
importlib.invalidate_caches()
piconewton_v4 = importlib.import_module('piconewton_v4')
from piconewton_v4.calibration import load_parameterization
from piconewton_v4.hypotheses import DecisionThresholds, write_decisions
from piconewton_v4.reporting import generate_standard_figures
from piconewton_v4.validation import validate_output_directory
from piconewton_v4.workflow import run_workflow

if piconewton_v4.__version__ != '4.0.0':
    raise RuntimeError(f'Unexpected package version: {piconewton_v4.__version__}')
module_path = Path(piconewton_v4.__file__).resolve()
if SOURCE_DIR.resolve() not in module_path.parents:
    raise RuntimeError(f'Package imported from an unexpected location: {module_path}')

_ = (RUN_DIR / 'provenance' / 'git_commit.txt').write_text(GIT_SHA + '\n', encoding='utf-8')
_ = (RUN_DIR / 'provenance' / 'package_import.txt').write_text(str(module_path) + '\n', encoding='utf-8')
print('Imported piconewton_v4:', piconewton_v4.__version__)
print('Import path:', module_path)


## Freeze scientific inputs
Before solving, the notebook confirms that the model configuration, calibration, and artery table exist and that the table contains exactly the required six arteries. These are input-integrity checks, not a preliminary simulation.


In [ ]:
import pandas as pd

CALIBRATION_PATH = PACKAGE_DIR / CALIBRATION_RELATIVE_PATH
MODEL_CONFIG_PATH = PACKAGE_DIR / 'configs' / 'model.json'
ARTERY_INPUT_PATH = PACKAGE_DIR / 'data' / 'ground_truth_arteries.csv'

for path in (CALIBRATION_PATH, MODEL_CONFIG_PATH, ARTERY_INPUT_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

arteries = pd.read_csv(ARTERY_INPUT_PATH)
artery_column = next(
    (column for column in arteries.columns if column.strip().lower() in {'artery', 'artery_name', 'name'}),
    None,
)
if artery_column is None:
    raise ValueError('No artery-name column was found in the six-artery input table.')
actual_arteries = set(arteries[artery_column].astype(str).str.strip())
if actual_arteries != EXPECTED_ARTERIES:
    raise ValueError(
        f'Six-artery input contract failed. Expected {sorted(EXPECTED_ARTERIES)}, '
        f'found {sorted(actual_arteries)}.'
    )

model_configuration = json.loads(MODEL_CONFIG_PATH.read_text(encoding='utf-8'))
interface_parameters, endpoint_parameters, calibration_audit = load_parameterization(CALIBRATION_PATH)

for source, destination in (
    (ARTERY_INPUT_PATH, RUN_DIR / 'inputs' / ARTERY_INPUT_PATH.name),
    (MODEL_CONFIG_PATH, RUN_DIR / 'configs' / MODEL_CONFIG_PATH.name),
    (CALIBRATION_PATH, RUN_DIR / 'configs' / CALIBRATION_PATH.name),
):
    shutil.copy2(source, destination)

input_record = {
    'run_id': RUN_ID,
    'git_sha': GIT_SHA,
    'package_version': piconewton_v4.__version__,
    'profile': NUMERICAL_PROFILE,
    'sensitivity_scan': RUN_SENSITIVITY_SCAN,
    'arteries': sorted(EXPECTED_ARTERIES),
    'calibration_status': calibration_audit.get('calibration_status'),
    'calibration_complete': calibration_audit.get('complete'),
    'missing_source_groups': calibration_audit.get('missing_source_groups', []),
}
_ = (RUN_DIR / 'provenance' / 'input_record.json').write_text(
    json.dumps(input_record, indent=2, sort_keys=True, default=str) + '\n',
    encoding='utf-8',
)
print(json.dumps(input_record, indent=2, sort_keys=True, default=str))


## Solve the production model
This is the only numerical solution cell. It runs all six arteries at the fixed publication resolution and executes the configured sensitivity analysis.


In [ ]:
if LOCAL_OUTPUT.exists():
    raise RuntimeError(
        f'Output directory already exists: {LOCAL_OUTPUT}. '
        'Start a fresh Runtime → Run all execution to preserve an immutable run.'
    )

try:
    manifest = run_workflow(
        package_root=PACKAGE_DIR,
        output_root=LOCAL_OUTPUT,
        profile=NUMERICAL_PROFILE,
        calibration_path=CALIBRATION_PATH,
        run_scan=RUN_SENSITIVITY_SCAN,
    )
except Exception as exc:
    failure_record = {
        'run_id': RUN_ID,
        'git_sha': GIT_SHA,
        'stage': 'run_workflow',
        'exception_type': type(exc).__name__,
        'exception': str(exc),
        'failed_utc': datetime.now(timezone.utc).isoformat(),
    }
    _ = (RUN_DIR / 'provenance' / 'failure.json').write_text(
        json.dumps(failure_record, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
    if LOCAL_OUTPUT.exists():
        shutil.copytree(LOCAL_OUTPUT, RUN_DIR / 'partial_model_outputs', dirs_exist_ok=True)
    raise

EXPECTED_WORKFLOW_STATUS = 'passed_structural_validation'
if manifest.get('status') != EXPECTED_WORKFLOW_STATUS:
    raise RuntimeError(
        f'Unexpected workflow status: {manifest.get("status")!r}; '
        f'expected {EXPECTED_WORKFLOW_STATUS!r}.'
    )
print('Workflow status:', manifest['status'])
print('Local output:', LOCAL_OUTPUT)


## Validate and classify the completed result
The completed outputs are checked for six-artery completeness, probability conservation, passive mechanics, and correct separation of signed force from exposure. Detection thresholds are then applied and the standard figures are generated.


In [ ]:
validation_report = validate_output_directory(LOCAL_OUTPUT)
if not validation_report.get('passed'):
    raise RuntimeError(validation_report)

thresholds = DecisionThresholds(
    current_rms_pa=endpoint_parameters.current_detection_limit_pa,
    calcium_rms_nm=endpoint_parameters.calcium_detection_limit_nm,
    minimum_arteries=MINIMUM_PASSING_ARTERIES,
)
decisions = write_decisions(
    LOCAL_OUTPUT / 'hypothesis_effects.csv',
    LOCAL_OUTPUT / 'hypothesis_decisions.csv',
    thresholds,
    LOCAL_OUTPUT / 'decision_thresholds.json',
)
figure_paths = generate_standard_figures(LOCAL_OUTPUT, LOCAL_OUTPUT / 'figures')
_ = (LOCAL_OUTPUT / 'output_validation.json').write_text(
    json.dumps(validation_report, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

print(json.dumps(validation_report, indent=2, sort_keys=True))
print('Decision rows:', len(decisions))
print('Figures generated:', len(figure_paths))


## Principal tables
The artery summary, effect sizes, and hypothesis decisions are displayed below. Complete arrays and all generated files are archived.


In [ ]:
from IPython.display import display

for name in (
    'six_artery_summary.csv',
    'hypothesis_effects.csv',
    'hypothesis_decisions.csv',
):
    path = LOCAL_OUTPUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(name)
    display(pd.read_csv(path))


## Archive and checksum
The complete native output tree, Python environment, run manifest, and SHA-256 checksums are stored in the timestamped Google Drive directory.


In [ ]:
_ = (RUN_DIR / 'environment' / 'pip_freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True),
    encoding='utf-8',
)

system_record = {
    'run_id': RUN_ID,
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'package_version': piconewton_v4.__version__,
    'package_import_path': str(Path(piconewton_v4.__file__).resolve()),
    'git_sha': GIT_SHA,
    'profile': NUMERICAL_PROFILE,
    'sensitivity_scan': RUN_SENSITIVITY_SCAN,
    'workflow_status': manifest.get('status'),
    'validation_passed': validation_report.get('passed'),
    'figure_count': len(figure_paths),
}
_ = (RUN_DIR / 'environment' / 'system.json').write_text(
    json.dumps(system_record, indent=2, sort_keys=True, default=str) + '\n',
    encoding='utf-8',
)

persistent_output = RUN_DIR / 'model_outputs'
if persistent_output.exists():
    raise RuntimeError(f'Persistent output unexpectedly exists: {persistent_output}')
shutil.copytree(LOCAL_OUTPUT, persistent_output)

_ = (RUN_DIR / 'provenance' / 'run_manifest.json').write_text(
    json.dumps(
        {
            **system_record,
            'arteries': sorted(EXPECTED_ARTERIES),
            'calibration_file': CALIBRATION_RELATIVE_PATH,
        },
        indent=2,
        sort_keys=True,
        default=str,
    ) + '\n',
    encoding='utf-8',
)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

checksums = {
    str(path.relative_to(RUN_DIR)): sha256(path)
    for path in sorted(RUN_DIR.rglob('*'))
    if path.is_file()
}
_ = (RUN_DIR / 'provenance' / 'SHA256SUMS.json').write_text(
    json.dumps(checksums, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

print('Production run complete:', RUN_ID)
print('Persistent output:', RUN_DIR)
print('Files checksummed:', len(checksums))
